In [5]:
def parse_whatsapp_chat(file_path):
    parsed_data = []
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            
            if " - " in line[:25] and ", " in line[:25]:
                
                time_part, rest_of_line = line.split(" - ", 1)
                
                
                if ": " in rest_of_line:
                    sender, message = rest_of_line.split(": ", 1)
                    
                    parsed_data.append({
                        'timestamp': time_part.strip(),
                        'sender': sender.strip(),
                        'text': message.strip()
                    })
                else:
                    
                    continue
            else:
                
                if parsed_data:
                    parsed_data[-1]['text'] += " " + line

    return parsed_data

chat_data = parse_whatsapp_chat('hostel_bois.txt')

print(f"Successfully loaded {len(chat_data)} messages!\n")
print("Here are the first 3 messages to verify:")
for i in range(3):
    print(chat_data[i])

Successfully loaded 3174 messages!

Here are the first 3 messages to verify:
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene'}


In [6]:
def get_overview_and_leaderboard(chat_data):
    total_messages = len(chat_data)
    
    first_date = chat_data[0]['timestamp'] if chat_data else "Unknown"
    last_date = chat_data[-1]['timestamp'] if chat_data else "Unknown"
   
    sender_counts = {}
    for msg in chat_data:
        sender = msg['sender']
        sender_counts[sender] = sender_counts.get(sender, 0) + 1
        
    
    leaderboard = sorted(sender_counts.items(), key=lambda x: x[1], reverse=True)
    
    return total_messages, first_date, last_date, leaderboard

def get_top_words(chat_data, top_n=10):
    stop_words = {
        'the', 'to', 'and', 'a', 'in', 'it', 'is', 'i', 'that', 'on', 'for', 
        'of', 'or', 'with', 'at', 'this', 'but', 'are', 'not', 'we', 'you', 
        'have', 'be', 'what', 'so', 'my', 'me', 'just', 'like', 'hai', 'ki', 
        'ko', 'se', 'bhi', 'aur', 'toh', 'ka', 'ke', 'mein', 'pe', 'kya', 'na'
    }
    
    word_counts = {}
    for msg in chat_data:
        text = msg['text'].lower()
        
        if "<media omitted>" in text or "this message was deleted" in text:
            continue
        
        for char in ['.', ',', '?', '!', ':', ';', '"', "'", '(', ')', '\n']:
            text = text.replace(char, ' ')
            
        words = text.split()
        for word in words:
        
            if word not in stop_words and len(word) > 2:
                word_counts[word] = word_counts.get(word, 0) + 1
                
    # Sort and return the top N words
    return sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]


total, start_d, end_d, leaders = get_overview_and_leaderboard(chat_data)
top_vocab = get_top_words(chat_data, top_n=5)

print(f"Date Range: {start_d} to {end_d}")
print(f"Total Messages: {total}\n")

print("--- LEADERBOARD ---")
for rank, (person, count) in enumerate(leaders, 1):
    print(f"{rank}. {person}: {count} messages")

print("\n--- TOP WORDS ---")
for word, count in top_vocab:
    print(f"'{word}': {count} times")

Date Range: 01/04/24, 01:17 to 30/05/24, 23:31
Total Messages: 3174

--- LEADERBOARD ---
1. Rahul: 953 messages
2. Priya: 718 messages
3. Neha: 635 messages
4. Aman: 490 messages
5. Karan: 354 messages
6. Vikas: 24 messages

--- TOP WORDS ---
'was': 385 times
'how': 321 times
'guys': 318 times
'today': 292 times
'about': 274 times


In [7]:
import numpy as np
from datetime import datetime

def generate_heatmap(chat_data, leaders, top_n=6):
    top_people = [person for person, count in leaders[:top_n]]
    
    heatmap_matrix = np.zeros((len(top_people), 24))
    
    for msg in chat_data:
        sender = msg['sender']
        if sender in top_people:
            
            try:
                time_obj = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
                hour = time_obj.hour
                
                
                row_idx = top_people.index(sender)
                
                
                heatmap_matrix[row_idx, hour] += 1
            except ValueError:
                
                continue
                
    return top_people, heatmap_matrix

def print_ascii_heatmap(top_people, heatmap_matrix):
    
    shades = [' ', '░', '▒', '▓', '█']
    
    print("\n--- HOURLY ACTIVITY HEATMAP ---")
    print(f"{'Name':<10} | 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23")
    print("-" * 80)
    
    for i, person in enumerate(top_people):
        row_data = heatmap_matrix[i]
        max_val = np.max(row_data)
        
       
        if max_val == 0:
            max_val = 1 
            
        
        row_str = f"{person[:10]:<10} | "
        
        for hour in range(24):
           
            intensity = row_data[hour] / max_val
            
            
            if intensity == 0:
                shade = shades[0]
            elif intensity < 0.25:
                shade = shades[1]
            elif intensity < 0.5:
                shade = shades[2]
            elif intensity < 0.75:
                shade = shades[3]
            else:
                shade = shades[4]
                
            row_str += shade + "  " 
            
        print(row_str)


top_users, activity_matrix = generate_heatmap(chat_data, leaders)
print_ascii_heatmap(top_users, activity_matrix)


--- HOURLY ACTIVITY HEATMAP ---
Name       | 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
--------------------------------------------------------------------------------
Rahul      | ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Priya      |                   ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒  ▒  ░  
Neha       |                ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒  
Aman       | ▓  █  █  ▓  █                             ░  ░  ░  ░  ░  ░  ░  ░     ▓  
Karan      |                      ░  ▒  ▒  ▓  ▒  █  ▓  █  ▓  ▓  ▓  ▓  █  ▓  ▒  ░  ░  
Vikas      |                      ▒  █  ▒  ▒     ▓  ▓     ▒  ▒  █  ▓  ▓  ▒  ▒  ▒  ▓  


In [8]:
def calculate_silent_streaks(chat_data, top_people):
    
    user_dates = {person: set() for person in top_people}
    
    for msg in chat_data:
        sender = msg['sender']
        if sender in top_people:
            try:
                
                time_obj = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
                user_dates[sender].add(time_obj.date())
            except ValueError:
                continue
                
    longest_silence = {}
    
    for person, dates in user_dates.items():
        if len(dates) < 2:
           
            longest_silence[person] = 0
            continue
            
        
        sorted_dates = sorted(list(dates))
        max_gap = 0
        
       
        for i in range(len(sorted_dates) - 1):
            
            gap = (sorted_dates[i+1] - sorted_dates[i]).days
            if gap > max_gap:
                max_gap = gap
                
        longest_silence[person] = max_gap
        
    return longest_silence


silence_stats = calculate_silent_streaks(chat_data, top_users)

print("\n--- LONGEST GHOSTING STREAKS ---")
for person, days in silence_stats.items():
    print(f"{person}: Went dark for {days} days max")


--- LONGEST GHOSTING STREAKS ---
Rahul: Went dark for 1 days max
Priya: Went dark for 1 days max
Neha: Went dark for 1 days max
Aman: Went dark for 1 days max
Karan: Went dark for 1 days max
Vikas: Went dark for 12 days max


In [9]:
from datetime import datetime

def assign_official_archetypes(chat_data, leaders):
    
    stats = {person: {'messages': 0, 'bursts': [], 'caring_words': 0, 'night_msgs': 0, 
                      'words': 0, 'drama_msgs': 0, 'active_days': set()} 
             for person, _ in leaders}
    
    
    caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
    
    current_sender = None
    current_burst = 0
    
    
    for msg in chat_data:
        sender = msg['sender']
        text = msg['text']
        
        if sender not in stats: 
            continue
            
        stats[sender]['messages'] += 1
        
       
        try:
            time_obj = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
            stats[sender]['active_days'].add(time_obj.date())
            
            if time_obj.hour >= 23 or time_obj.hour <= 4:
                stats[sender]['night_msgs'] += 1
        except ValueError:
            pass
            
        
        stats[sender]['words'] += len(text.split())
        
        
        if "<media" not in text.lower() and "deleted" not in text.lower():
            if (text.isupper() and len(text) > 2) or text.count('!') >= 2:
                stats[sender]['drama_msgs'] += 1
                
       
        text_lower = text.lower()
        for kw in caring_keywords:
            if kw in text_lower:
                stats[sender]['caring_words'] += 1
                
       
        if sender == current_sender:
            current_burst += 1
        else:
            if current_sender in stats and current_burst > 0:
                stats[current_sender]['bursts'].append(current_burst)
            current_sender = sender
            current_burst = 1
            
    
    if current_sender in stats:
        stats[current_sender]['bursts'].append(current_burst)
        
   
    archetypes = {}
    for person in stats:
        p_stats = stats[person]
        total_msgs = p_stats['messages']
        if total_msgs == 0: 
            continue
        
       
        avg_burst = sum(p_stats['bursts']) / len(p_stats['bursts']) if p_stats['bursts'] else 0
        caring_score = p_stats['caring_words']
        night_ratio = p_stats['night_msgs'] / total_msgs
        avg_words = p_stats['words'] / total_msgs
        drama_ratio = p_stats['drama_msgs'] / total_msgs
        silent_days = 60 - len(p_stats['active_days'])
        
        
        if silent_days >= 36: 
            archetypes[person] = {"title": "THE GHOST", "desc": f"(silent on {silent_days} of 60 days)"}
        elif night_ratio > 0.60:
            archetypes[person] = {"title": "THE NIGHT OWL", "desc": f"({night_ratio*100:.1f}% msgs between 23h-04h)"}
        elif drama_ratio > 0.30:
            archetypes[person] = {"title": "THE DRAMA QUEEN", "desc": f"({drama_ratio*100:.1f}% ALL-CAPS messages)"}
        elif avg_words > 30:
            archetypes[person] = {"title": "THE STORYTELLER", "desc": f"(avg {avg_words:.1f} words per msg)"}
        elif avg_burst > 3:
            archetypes[person] = {"title": "THE SPAMMER", "desc": f"(avg {avg_burst:.1f} msgs in a row)"}
        else:
           
            archetypes[person] = {"title": "THE GROUP MOM", "desc": f"(caring keyword score: {caring_score})"}
            
    return archetypes

chat_data = parse_whatsapp_chat('hostel_bois.txt')


total, start_d, end_d, leaders = get_overview_and_leaderboard(chat_data)

official_archetypes = assign_official_archetypes(chat_data, leaders)


print("\nPERSONALITY ARCHETYPES")
for person, data in official_archetypes.items():
    print(f"{person:<10} -> {data['title']:<20} {data['desc']}")



PERSONALITY ARCHETYPES
Rahul      -> THE SPAMMER          (avg 4.5 msgs in a row)
Priya      -> THE GROUP MOM        (caring keyword score: 621)
Neha       -> THE DRAMA QUEEN      (62.2% ALL-CAPS messages)
Aman       -> THE NIGHT OWL        (79.8% msgs between 23h-04h)
Karan      -> THE STORYTELLER      (avg 55.7 words per msg)
Vikas      -> THE GHOST            (silent on 44 of 60 days)


In [10]:
def get_extra_time_stats(chat_data, leaders):
    day_counts = {}
    hour_counts = {}
    response_gaps = {person: [] for person, _ in leaders}
    
    last_sender = None
    last_time = None
    
    for msg in chat_data:
        try:
            t = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
            
    
            day_str = t.strftime('%d %B %Y')
            day_counts[day_str] = day_counts.get(day_str, 0) + 1
            hour_counts[t.hour] = hour_counts.get(t.hour, 0) + 1
            
            
            if last_sender and last_sender != msg['sender'] and last_time:
                gap_minutes = (t - last_time).total_seconds() / 60
                
                if gap_minutes < (12 * 60): 
                    response_gaps[msg['sender']].append(gap_minutes)
                    
            last_sender = msg['sender']
            last_time = t
        except ValueError:
            pass
            
    busiest_day = max(day_counts.items(), key=lambda x: x[1])
    busiest_hour = max(hour_counts.items(), key=lambda x: x[1])
    
    avg_responses = {}
    for p, gaps in response_gaps.items():
        if gaps:
            avg_responses[p] = sum(gaps) / len(gaps)
            
    fastest = min(avg_responses.items(), key=lambda x: x[1])
    slowest = max(avg_responses.items(), key=lambda x: x[1])
    
    return busiest_day, busiest_hour, fastest, slowest


def print_final_report(chat_data, leaders, top_vocab, top_users, activity_matrix, silence_stats, official_archetypes):
    total_msgs = len(chat_data)
    first_date = chat_data[0]['timestamp'][:8]
    last_date = chat_data[-1]['timestamp'][:8]
    
    b_day, b_hour, fastest, slowest = get_extra_time_stats(chat_data, leaders)
    
    print("\n" + "="*60)
    print(f"{'GROUPDNA REPORT':^60}")
    print("="*60)
    print(f"{'Group':<15}: \"Hostel Bois 4ever\"")
    print(f"{'Period':<15}: {first_date} to {last_date} (60 days)")
    print(f"{'Total messages':<15}: {total_msgs:,}")
    print(f"{'Participants':<15}: {len(leaders)}")
    print("="*60)
    
    print(f"\nBusiest day    : {b_day[0]} ({b_day[1]} messages)")
    print(f"Busiest hour   : {b_hour[0]:02d}:00 - {b_hour[0]+1:02d}:00\n")
    
    print("MESSAGES PER PERSON")
    for person, count in leaders:
        percentage = (count / total_msgs) * 100
        print(f"{person:<10} : {count:>4} ({percentage:>4.1f}%)")
        
    
    print_ascii_heatmap(top_users, activity_matrix)
    
    print("\nTHIS GROUP'S FAVOURITE WORDS")
    for word, count in top_vocab:
        
        bar_length = int((count / top_vocab[0][1]) * 15)
        bar = "█" * bar_length
        print(f"{word:<10} {bar} {count}")
        
    print("\nRESPONSE PATTERNS")
    print(f"Fastest replier: {fastest[0]} (avg {fastest[1]:.1f} minutes)")
    
    
    slow_hours = slowest[1] / 60
    print(f"Slowest replier: {slowest[0]} (avg {slow_hours:.1f} hours)\n")
    
    print("LONGEST SILENT STREAKS")
   
    sorted_silence = sorted(silence_stats.items(), key=lambda x: x[1], reverse=True)
    for person, days in sorted_silence:
        if days > 0:
            print(f"{person:<10} : {days} days")
            
    print("\nPERSONALITY ARCHETYPES")
    for person, data in official_archetypes.items():
        print(f"{person:<10} -> {data['title']:<20} {data['desc']}")
        
    print("\n" + "="*60)
    print(f"{'Generated by GroupDNA':^60}")
    print(f"{'Built with Python + NumPy':^60}")
    print("="*60 + "\n")


print_final_report(chat_data, leaders, top_vocab, top_users, activity_matrix, silence_stats, official_archetypes)


                      GROUPDNA REPORT                       
Group          : "Hostel Bois 4ever"
Period         : 01/04/24 to 30/05/24 (60 days)
Total messages : 3,174
Participants   : 6

Busiest day    : 04 May 2024 (76 messages)
Busiest hour   : 18:00 - 19:00

MESSAGES PER PERSON
Rahul      :  953 (30.0%)
Priya      :  718 (22.6%)
Neha       :  635 (20.0%)
Aman       :  490 (15.4%)
Karan      :  354 (11.2%)
Vikas      :   24 ( 0.8%)

--- HOURLY ACTIVITY HEATMAP ---
Name       | 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
--------------------------------------------------------------------------------
Rahul      | ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Priya      |                   ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒  ▒  ░  
Neha       |                ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒  
Aman       | ▓  █  █  ▓  █                             ░  ░  ░  ░  ░  ░  ░  ░     ▓  
Karan      |    